In [ ]:
folder_url="https://drive.google.com/drive/folders/1ACcHWvQ7hLmDc2rimAaQwH_OF4bPoTnS?usp=drive_link"

import os
os.environ["folder_url"] = folder_url

In [ ]:
%%bash
mkdir /kaggle/working/assets
gdown --folder $folder_url --output /kaggle/working/assets

In [ ]:
import cv2
import os
from pathlib import Path

def extract_frames(video_path, output_dir='frame_extraction', target_fps=15):
    """Extract frames from video at specified fps and return list of frame paths"""
    
    # Create output directory if it doesn't exist
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    filename = os.path.basename(video_path)
    # Open video
    cap = cv2.VideoCapture(video_path)
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = max(1, round(video_fps / target_fps))
    
    frame_paths = []
    frame_count = 0
    saved_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Save frame at specified interval
        if frame_count % frame_interval == 0:
            frame_path = os.path.join(output_dir, f"{filename}_{saved_count:06d}.png")
            cv2.imwrite(frame_path, frame)
            frame_paths.append(frame_path)
            saved_count += 1
            print(f"Saved: {filename}_{saved_count:06d}.png")
        
        frame_count += 1
    
    cap.release()
    print(f"Extracted {saved_count} frames from {video_path}")
    return frame_paths


In [ ]:
import glob
dav_files = glob.glob("assets/*.dav")
print(f"Found {len(dav_files)} DAV files: {dav_files}")


In [ ]:
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!chmod +x miniconda.sh
!bash miniconda.sh -b -p /miniconda

!. /miniconda/etc/profile.d/conda.sh && conda init
!. /miniconda/etc/profile.d/conda.sh && conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main && conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!. /miniconda/etc/profile.d/conda.sh && conda create -n my_env python=3.12 -y && conda activate my_env

In [ ]:
%%writefile on_conda
#!/bin/bash

src=". /miniconda/etc/profile.d/conda.sh"
env="conda activate my_env"
cmd="$src && $env && $@"
eval $cmd

In [ ]:
!chmod -x on_conda

import os
os.environ["konda"] = "bash on_conda"
!$konda conda --version

In [ ]:
!git clone https://github.com/ByteDance-Seed/Depth-Anything-3.git

In [ ]:
!$konda pip install xformers torch\>=2 torchvision -q
!$konda pip install -e Depth-Anything-3/. -q # Basic 
!$konda pip install --no-build-isolation git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70 -q# for gaussian head
!$konda pip install -e "Depth-Anything-3/.[app]" -q # Gradio, python>=3.10
!$konda pip install -e "Depth-Anything-3/.[all]" -q # ALL

In [ ]:
# Install required packages
!pip install pyngrok fastapi uvicorn flask -q


In [ ]:
# === Flask Web Terminal (call web_print(...) from cells below) ===
from flask import Flask, jsonify, render_template_string
from werkzeug.serving import make_server
from threading import Thread, Lock
from queue import Queue, Empty
from collections import deque
from datetime import datetime
import socket

# Persist these across cell re-runs
if 'WEB_PRINT_QUEUE' not in globals():
    WEB_PRINT_QUEUE = Queue()
if 'WEB_TERMINAL_BUFFER' not in globals():
    WEB_TERMINAL_BUFFER = deque(maxlen=2000)
if 'WEB_TERMINAL_LOCK' not in globals():
    WEB_TERMINAL_LOCK = Lock()

def web_print(msg):
    ts = datetime.now().strftime('%H:%M:%S')
    text = '' if msg is None else str(msg)
    lines = text.splitlines() or ['']
    for line in lines:
        WEB_PRINT_QUEUE.put(f'[{ts}] {line}')

def _drain_web_queue(max_items=500):
    drained = 0
    while drained < max_items:
        try:
            line = WEB_PRINT_QUEUE.get_nowait()
        except Empty:
            break
        with WEB_TERMINAL_LOCK:
            WEB_TERMINAL_BUFFER.append(line)
        drained += 1
    return drained

def _is_port_free(host, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.2)
        return sock.connect_ex((host, port)) != 0

def _pick_port(host='127.0.0.1', preferred=5000):
    if _is_port_free(host, preferred):
        return preferred
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind((host, 0))
        return int(sock.getsockname()[1])

HOME_HTML = '''
<!doctype html>
<title>Web Terminal</title>
<style>
body { font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; margin: 20px; }
#terminal {
  background: #0b0f0b;
  color: #d1f7c4;
  border: 1px solid #1f2a1f;
  border-radius: 8px;
  padding: 12px;
  height: 360px;
  overflow-y: auto;
  white-space: pre-wrap;
  font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace;
  font-size: 13px;
  line-height: 1.35;
}
.small { color: #666; font-size: 12px; }
</style>

<h1>Web Terminal</h1>
<p class='small'>From a notebook cell, call <code>web_print('hello')</code> to push messages here.</p>
<pre id='terminal'>Loading…</pre>

<script>
async function refreshTerminal() {
  const el = document.getElementById('terminal');
  try {
    const res = await fetch('/api/terminal');
    const data = await res.json();
    const shouldStick = (el.scrollTop + el.clientHeight + 30) >= el.scrollHeight;
    el.textContent = data.text || '';
    if (shouldStick) { el.scrollTop = el.scrollHeight; }
  } catch (e) {
    el.textContent = 'Error loading terminal: ' + e;
  }
}
setInterval(refreshTerminal, 1000);
window.addEventListener('load', refreshTerminal);
</script>
'''

if 'web_app' not in globals():
    web_app = Flask('web_terminal')

@web_app.get('/')
def web_home():
    _drain_web_queue()
    return render_template_string(HOME_HTML)

@web_app.get('/api/terminal')
def web_terminal_api():
    _drain_web_queue(max_items=2000)
    with WEB_TERMINAL_LOCK:
        text = '\n'.join(WEB_TERMINAL_BUFFER)
    return jsonify(text=text, lines=len(WEB_TERMINAL_BUFFER))

class _WebServerThread(Thread):
    def __init__(self, app, host, port):
        super().__init__(daemon=True)
        self._server = make_server(host, port, app)

    def run(self):
        self._server.serve_forever()

    def shutdown(self):
        self._server.shutdown()

# If you re-run this cell, stop the previous server first.
if 'web_server' in globals():
    try:
        web_server.shutdown()
    except Exception:
        pass

WEB_HOST = '127.0.0.1'
WEB_PORT = _pick_port(WEB_HOST, preferred=5000)

web_server = _WebServerThread(web_app, WEB_HOST, WEB_PORT)
web_server.start()

web_print('Web terminal started.')
print(f'Web terminal: http://{WEB_HOST}:{WEB_PORT}')


In [ ]:
!mkdir output

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token

In [ ]:
import threading
import time
import subprocess
import sys
import select
import os
from pyngrok import ngrok

from queue import Queue
from collections import deque
from itertools import combinations

# === CONFIGURATION ===

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("ngrok")

NGROK_AUTH_TOKEN = token  # Replace with your token
LOCAL_PORT = 8008  # Port your local service runs on

L = 4

# === FUNCTION FOR FIRST SUBPROCESS ===
def run_first_process():
    """Run the first subprocess to cache model to GPU"""
    print("🚀 Thread 1: Starting first process - Caching model to GPU...")
    try:
        # First subprocess: cache model to GPU
        process1 = subprocess.Popen(
            ["bash","on_conda", "da3", "backend", "--model-dir", "depth-anything/DA3NESTED-GIANT-LARGE", "--gallery-dir", "gallery"],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            stdin=subprocess.PIPE,
            universal_newlines=True,
            bufsize=1
        )
        
        # Print output in real-time
        for line in process1.stdout:
            print(f"[Thread 1 - Process 1] {line.strip()}")
        
        # Wait for process to complete
        process1.wait()
            
    except Exception as e:
        print(f"❌ Thread 1: Error in first process: {e}")
        first_process_completed = False

import secrets
import base64


def generate_unique_base64():
    """Generate a unique base64 string from 16 random bytes"""
    random_bytes = os.urandom(16)
    base64_string = base64.b64encode(random_bytes).decode('utf-8')
    return base64_string

task_flag = False

# Alternative simpler approach using expect-like pattern
def run_second_process_simple():

    global dav_files
    
    print("⏱️ Thread 2: First process completed. Waiting 10 seconds before starting...")
    time.sleep(10)
    
    print("🚀 Thread 2: Starting second process - Processing video assets...")
    try:
        # Generate Frames

        
    
        for dav in dav_files:
            frames = extract_frames(dav)
        

        # generate a queue for L combination frames
            global L
            combs = frames
    
            def start_task(comb):
    
                tmp = generate_unique_base64()
                print(f'Executing task for: {tmp}')
                print(f'File: {comb}')
                
                
                
                proccess = subprocess.Popen(
                    ["bash","on_conda","da3", "auto", f'{comb}', "--export-format", "glb-feat_vis", "--export-dir", f"output/{comb}", '--auto-cleanup','--use-backend'],
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE,
                    stdin=subprocess.PIPE,
                    universal_newlines=True,
                    bufsize=1
                )
    
                for line in proccess.stdout:
                    print(f'[Task - {tmp}] {line.strip()}')
                
                proccess.wait()
                print(f'Task: {tmp} is complete.')
    
                
    
            for comb in combs:
                try:
                    start_task(comb)
                except Exception as e:
                    print(e)

            global task_flag
            task_flag = True
    
    except Exception as e:
        print(e)
        




# === SETUP NGROK ===
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start ngrok tunnel
public_url = ngrok.connect(LOCAL_PORT, "http")
print(f"🌍 Public URL: {public_url}")
print("✅ Tunnel is active! Keep this cell running.")

# Global flag to coordinate between threads
first_process_completed = None  # None = not started, False = failed, True = completed


# === START BOTH SUBPROCESSES IN SEPARATE THREADS ===
thread1 = threading.Thread(target=run_first_process, daemon=True, name="Process1-Thread")
# Choose one of the second process implementations:
thread2 = threading.Thread(target=run_second_process_simple, daemon=True, name="Process2-Thread")  # More robust version
# thread2 = threading.Thread(target=run_second_process_simple, daemon=True, name="Process2-Thread")  # Simpler version

# Start both threads
thread1.start()
thread2.start()

print("🔄 Both background threads started (Thread 2 will wait for Thread 1 to complete + 10s)")
print("🤖 Thread 2 will automatically respond 'yes' to any confirmation prompts")

# === KEEP CELL ALIVE ===
try:
    while True:
        time.sleep(60)
        # Check if ngrok is still active
        tunnels = ngrok.get_tunnels()
        if not tunnels:
            print("⚠️ Tunnel died, restarting...")
            public_url = ngrok.connect(LOCAL_PORT, "http")
            print(f"🌍 New URL: {public_url}")
        else:
            # Check thread status
            if not thread1.is_alive() and not thread2.is_alive():
                print("✅ Both process threads have completed")
            elif not thread1.is_alive() and thread2.is_alive():
                print("🔄 Thread 1 completed, Thread 2 still running")
            elif thread1.is_alive() and not thread2.is_alive():
                print("🔄 Thread 2 completed, Thread 1 still running? (This shouldn't happen normally)")
            else:
                print(f"🔄 Both threads running - Thread 1: {thread1.is_alive()}, Thread 2: {thread2.is_alive()}")

        if task_flag:
            break
        
except KeyboardInterrupt:
    ngrok.kill()
    print("🛑 Tunnel closed")

In [ ]:
!zip -r -9 "DA3_$(date +%Y-%m-%d).zip" output/

In [ ]:
!pip install mega.py

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
email = user_secrets.get_secret("mega_email")
pasw = user_secrets.get_secret("mega_pass")


In [ ]:
from mega import Mega

mega = Mega()

# login
m = mega.login(email, pasw)

# upload file
file = m.upload("output.zip")

print("Uploaded:", file)